# sm'

In [ ]:
///- --real --test static

open testing

In [ ]:
///- --test static

open testing

In [ ]:
open rust
open rust_operators

///- --static

open sm'_real

## rust

### std_string_real

In [ ]:
///- --real

nominal std_string_real =
    `(
        rust_type
            `($'std_string_String')
            `($'std::string::String')
            "type std_string_String = string"
    )

### std_string

In [ ]:
type std_string = std_string_real

### to_string'

In [ ]:
inl to_string' forall t. (x : t) : std_string =
    !\($'$"!x.to_string()"')

### from_std_string_real

In [ ]:
///- --real

inl from_std_string_real (str : std_string_real) : string =
    open rust
    rust.emit_expr `std_string_real `string str ($'"Rc::<str>::from($0)"' : string)

### from_std_string

In [ ]:
inl from_std_string (str : std_string) : string =
    backend_switch {
        Rust = fun () => !\\(str, $'"Rc::<str>::from(String::as_str(&$0))"') : string
    }

## sm'

### symbol_to_string_real

In [ ]:
///- --real

inl symbol_to_string_real forall t {symbol}. : string =
    inl x = real_core.type_to_symbol `t
    !!!!SymbolToString (x)

### symbol_to_string

In [1]:
///- --real --test
///> gleam
///> lua
///> fsharp
///> cuda

open testing
inl x = .test
inl x = symbol_to_string_real `(`x)
_assert_eq `string "test" x

.gleam (gleam)
__assert_eq { expected = test }
.lua (lua)
__assert_eq { expected = test }
.fsx (fsharp)
__assert_eq { expected = test }
.py (cuda)
__assert_eq { expected = test }

In [ ]:
inl symbol_to_string forall t {symbol}. (x : t) : string =
    real symbol_to_string_real `t

In [2]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda

.test
|> symbol_to_string
|> _assert_eq "test"

.gleam (gleam)
__assert_eq { expected = test }
.lua (lua)
__assert_eq { expected = test }
.fsx (fsharp)
__assert_eq { expected = test }
.py (cuda)
__assert_eq { expected = test }

### index

In [ ]:
inl index i (str : string) : char =
    i |> sm.index str

### length

In [ ]:
inl length forall dim {int}. (input : string) : dim =
    input |> sm.length

In [3]:
///- --test
///> gleam
///> fsharp
///> cuda

"abc"
|> length
|> _assert_eq 3i32

.gleam (gleam)
__assert_eq { expected = 3 }
.fsx (fsharp)
__assert_eq { expected = 3 }
.py (cuda)
__assert_eq { expected = 3 }

### (~##)

In [ ]:
inl (~##) s : int =
    s |> length

### to_char_list

In [ ]:
inl to_char_list (str : string) : list char =
    listm.init (str |> length) (fun (i : i64) => str |> index i)

In [4]:
///- --test
///> gleam
///> fsharp
///> cuda

"abc"
|> to_char_list
|> _assert_eq [ 'a'; 'b'; 'c' ]

.gleam (gleam)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }

### to_char_array

In [ ]:
inl to_char_array (str : string) : array_base char =
    am'.init_base (str |> length) (fun i => str |> index i)

In [5]:
///- --test
///> lua
///> gleam
///> fsharp
///> cuda

"abc"
|> to_char_array
|> fun x => a x : _ int _
|> am'.to_list'
|> listm'.unbox
|> _assert_eq [ 'a'; 'b'; 'c' ]

.lua (lua)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.gleam (gleam)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }

### is_empty

In [ ]:
inl is_empty (input : string) : bool =
    (input |> length) = 0i32

### to_option

In [ ]:
inl to_option s =
    if s |> is_empty
    then None
    else s |> Some

### string_builder_real

In [ ]:
///- --real

nominal string_builder_python =
    `(
        global "import io"
        $'' : $'io.StringIO'
    )
type string_builder_switch =
    {
        Gleam : string
        Fsharp : $'System.Text.StringBuilder'
        Python : string_builder_python
        Rust : $'String'
        TypeScript : string
    }
nominal string_builder_real = $'backend_switch `(string_builder_switch)'

### string_builder

In [ ]:
type string_builder = string_builder_real

inl string_builder (initial : string) : string_builder =
    inl initial =
        if initial = ""
        then join initial
        else initial

    backend_switch {
        Gleam = fun () => $'!initial ' : string_builder
        Lua = fun () => $'!initial ' : string_builder
        Fsharp = fun () => initial |> $'`string_builder ' : string_builder
        Python = fun () =>
            global "import io"
            global "class CustomStringIO(io.StringIO):\n    def __init__(self, init=''):\n        super().__init__()\n        if init != '': self.write(init)\n    def __str__(self): return self.getvalue()\n    def __repr__(self): return self.getvalue()"
            $'CustomStringIO(!initial)' : string_builder
        Rust = fun () => $'String::from(&*!initial)' : string_builder
        TypeScript = fun () => $'!initial ' : string_builder
    }

### append_real

In [ ]:
///- --real

inl append_real (x : string) (s : string) : string =
    backend_switch `string `({}) {
        Gleam = (fun () => $'!s <> !x ' : string) : () -> string
        Lua = (fun () => $'!s .. !x ' : string) : () -> string
        Fsharp = (fun () => $'!s + !x ' : string) : () -> string
        Python = (fun () => $'!s + !x ' : string) : () -> string
        Rust = (fun () => $'Rc::<str>::from(format\!("{}{}", !s, !x))' : string) : () -> string
        TypeScript = (fun () => $'!s + !x ' : string) : () -> string
        Zig = (fun () => $'spiralConcat(!s, !x)' : string) : () -> string
        Lean = (fun () => $'(!s ++ !x)' : string) : () -> string
        C = (fun () => !!!!StaticStringConcat(s, x)) : () -> string
        Delphi = (fun () => $'!s + !x ' : string) : () -> string
        Bend = (fun () => !!!!StaticStringConcat(s, x)) : () -> string
    }

### append

In [ ]:
inl append (x : string) (s : string) : string =
    real append_real x s

### (++#*)

In [ ]:
///- --real

inl (++#*) (a : string) (b : string) : string =
    append_real b a

### (++#)

In [ ]:
inl (++#) (a : string) (b : string) : string =
    real (++#*) a b

### (++\\\\#)

In [ ]:
inl (++\\#) a b =
    a ++# "\\n" ++# b

### (++\\#)

In [ ]:
inl (++\#) a b =
    a ++# "\n" ++# b

### format_debug_real

In [ ]:
///- --real

inl rec format_debug_real forall t. (x : t) : string =
    typecase t with
    | string => x
    | _ =>
        backend_switch `string `({}) {
            Gleam =
                (fun () =>
                    typecase t with
                    | char => convert.to `(`x) `string x
                    | string_builder_real => convert.to `(`x) `string x
                    | _ =>
                        global "import gleam/string"
                        inl x =
                            typecase t with
                            | array_base _ => $'!x |> spiral_array_to_list' : t
                            | _ => x
                        $'string.inspect(!x)' : string
                    ) : () -> string
            Lua =
                (fun () =>
                    typecase t with
                    | ~a * ~b =>
                        inl a, b =
                            if not (var_is `t x)
                            then x
                            else ($'__v[1]._1' : a), ($'__v[1]._2' : b)
                        inl head_str = format_debug_real `a a
                        inl tail_str = format_debug_real `b b
                        (++#*) head_str ((++#*) ", " tail_str)
                    | {} => $'table.concat(!x, ", ")   ' : string
                    | array_base _ =>
                        (++#*) ((++#*) "[" ($'table.concat(!x, ", ")' : string)) "]"
                    | t =>
                        if real_core.union_type_is `t
                        then serialize_union `t x
                        else $'tostring(!x)       ' : string
                ) : () -> string
            Fsharp = (fun () => $'$"%A{!x}"' : string) : () -> string
            Python =
                (fun () =>
                    $'f"{!x}"' : string
                ) : () -> string
            Rust =
                (fun () =>
                    typecase t with
                    | char => $'Rc::<str>::from(format\!("{:?}", !x as char))' : string
                    | optionm'.option' ~a =>
                        if real_core.prim_type_is `a
                        then $'Rc::<str>::from(match &!x { Some(v) => format\!("Some {:?}", v), None => String::from("None") })' : string
                        else
                            inl x = $'Rc::new(RefCell::new(!x.clone().into_iter().collect::<Vec<_>>()))' : array_base a
                            if !!!!EQ(!!!!ArrayLength(`u64, x), 0u64) then "None"
                            else (++#*) "Some " (format_debug_element_rust `a !!!!ArrayIndex(x, 0u64))
                    | () * ~b =>
                        inl _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        inl a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | array_base ~el =>
                        if real_core.prim_type_is `el
                        then $'Rc::<str>::from(format\!("{:?}", !x.borrow()))' : string
                        else format_debug_elements_rust `el x
                    | listm'.list' ~el =>
                        if real_core.prim_type_is `el
                        then format_debug_other_rust `t x
                        else format_debug_elements_rust `el ($'Rc::new(RefCell::new((*!x).clone()))' : array_base el)
                    | seq.seq' ~el =>
                        if real_core.prim_type_is `el
                        then format_debug_other_rust `t x
                        else format_debug_elements_rust `el ($'Rc::new(RefCell::new((*!x).clone()))' : array_base el)
                    | {} =>
                        inl _, fields =
                            real_core.record_fold
                                fun { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ => format_debug_other_rust `t x
                ) : () -> string
            TypeScript =
                (fun () =>
                    typecase t with
                    | char => $'"\'" + String.fromCharCode(!x) + "\'"' : string
                    | optionm'.option' ~a =>
                        if $'!x === undefined' : bool
                        then "None"
                        else (++#*) "Some " (format_debug_real `a ($'!x ' : a))
                    | () * ~b =>
                        inl _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        inl a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | array_base ~el =>
                        if (typecase el with char => true | _ => false)
                        then $'"[" + !x.map((c: number) => "\'" + String.fromCharCode(c) + "\'").join(", ") + "]"' : string
                        elif real_core.prim_type_is `el
                        then $'"[" + !x.join(", ") + "]"' : string
                        else $'String(!x)' : string
                    | {} =>
                        inl _, fields =
                            real_core.record_fold
                                fun { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        if real_core.union_type_is `t
                        then serialize_union_rust `t x
                        elif real_core.nominal_type_is `t
                        then
                            inl y = real_core.nominal_strip x
                            format_debug_real `(`y) y
                        else $'String(!x)' : string
                ) : () -> string
            Zig =
                (fun () =>
                    typecase t with
                    | char => $'(std.fmt.allocPrint(spiral_gpa, "\'{c}\'", .{ !x }) catch \@panic("out of memory"))' : string
                    | bool => $'(if (!x) "true" else "false")' : string
                    | () * ~b =>
                        inl _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        inl a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        inl _, fields =
                            real_core.record_fold
                                fun { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        if real_core.union_type_is `t
                        then serialize_union_rust `t x
                        elif real_core.nominal_type_is `t
                        then
                            inl y = real_core.nominal_strip x
                            format_debug_real `(`y) y
                        elif real_core.prim_type_is `t
                        then $'(std.fmt.allocPrint(spiral_gpa, "{d}", .{ !x }) catch \@panic("out of memory"))' : string
                        else $'(std.fmt.allocPrint(spiral_gpa, "{any}", .{ !x }) catch \@panic("out of memory"))' : string
                ) : () -> string
            Delphi =
                (fun () =>
                    typecase t with
                    | bool => $'BoolToStr(!x, \'true\', \'false\')' : string
                    | char => $'string(!x)' : string
                    | f32 => $'FloatToStr(!x)' : string
                    | f64 => $'FloatToStr(!x)' : string
                    | () * ~b =>
                        inl _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        inl a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        inl _, fields =
                            real_core.record_fold
                                fun { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        if real_core.union_type_is `t
                        then serialize_union_rust `t x
                        elif real_core.nominal_type_is `t
                        then
                            inl y = real_core.nominal_strip x
                            format_debug_real `(`y) y
                        else $'IntToStr(!x)' : string
                ) : () -> string
            Lean =
                (fun () =>
                    typecase t with
                    | char => $'("\'" ++ toString !x ++ "\'")' : string
                    | () * ~b =>
                        inl _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        inl a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        inl _, fields =
                            real_core.record_fold
                                fun { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        if real_core.union_type_is `t
                        then serialize_union_rust `t x
                        elif real_core.nominal_type_is `t
                        then
                            inl y = real_core.nominal_strip x
                            format_debug_real `(`y) y
                        else $'toString !x ' : string
                ) : () -> string
            C =
                (fun () =>
                    inl buffer () =
                        global "#include <stdio.h>"
                        global "#include <stdint.h>"
                        global "#include <stdlib.h>"
                        global "#include <string.h>"
                        global "static char spiral_format_buffer[64];"
                    inl signed x : string =
                        buffer ()
                        global "static inline uint32_t spiral_format_i64(long long x) { return (uint32_t)snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"%lld\", x) + 1u; }"
                        $'StringLit(spiral_format_i64((long long)(!x)), spiral_format_buffer)' : string
                    inl unsigned x : string =
                        buffer ()
                        global "static inline uint32_t spiral_format_u64(unsigned long long x) { return (uint32_t)snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"%llu\", x) + 1u; }"
                        $'StringLit(spiral_format_u64((unsigned long long)(!x)), spiral_format_buffer)' : string
                    inl decimal_point () =
                        global "static inline uint32_t spiral_format_decimal_point(int n) { if (!strpbrk(spiral_format_buffer, \".eni\")) { spiral_format_buffer[n++] = '.'; spiral_format_buffer[n++] = '0'; spiral_format_buffer[n] = 0; } return (uint32_t)n + 1u; }"
                    inl float64 (x : f64) : string =
                        buffer ()
                        decimal_point ()
                        global "static inline uint32_t spiral_format_f64(double x) { int n = 0; for (int p = 1; p <= 17; p++) { n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"%.*g\", p, x); if (strtod(spiral_format_buffer, NULL) == x) break; } return spiral_format_decimal_point(n); }"
                        $'StringLit(spiral_format_f64(!x), spiral_format_buffer)' : string
                    inl float32 (x : f32) : string =
                        buffer ()
                        decimal_point ()
                        global "static inline uint32_t spiral_format_f32(float x) { int n = 0; for (int p = 1; p <= 9; p++) { n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"%.*g\", p, (double)x); if (strtof(spiral_format_buffer, NULL) == x) break; } return spiral_format_decimal_point(n); }"
                        $'StringLit(spiral_format_f32(!x), spiral_format_buffer)' : string
                    typecase t with
                    | char =>
                        buffer ()
                        global "static inline uint32_t spiral_format_char(int32_t x) { return (uint32_t)snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"'%c'\", (int)x) + 1u; }"
                        $'StringLit(spiral_format_char(!x), spiral_format_buffer)' : string
                    | bool => if x then "true" else "false"
                    | f64 => float64 x
                    | f32 => float32 x
                    | i8 => signed x
                    | i16 => signed x
                    | i32 => signed x
                    | i64 => signed x
                    | u8 => unsigned x
                    | u16 => unsigned x
                    | u32 => unsigned x
                    | u64 => unsigned x
                    | array_base ~el => format_debug_elements_rust `el x
                    | () * ~b =>
                        inl _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        inl a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        inl _, fields =
                            real_core.record_fold
                                fun { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        if real_core.union_type_is `t
                        then serialize_union_rust `t x
                        elif real_core.nominal_type_is `t
                        then
                            inl y = real_core.nominal_strip x
                            format_debug_real `(`y) y
                        else $'StringLit(sizeof("<`t>"), "<`t>")' : string
                ) : () -> string
        }
and inl format_debug_other_rust forall t. (x : t) : string =
    if real_core.prim_type_is `t
    then $'Rc::<str>::from(format\!("{:?}", !x))' : string
    elif real_core.union_type_is `t
    then serialize_union_rust `t x
    elif real_core.nominal_type_is `t
    then
        inl payload forall u. (y : u) : string =
            typecase u with
            | {} => format_debug_real `u y
            | ~a -> ~b => $'Rc::<str>::from("<`t>")' : string
            | ~a * ~b => format_debug_real `u y
            | array_base _ => format_debug_real `u y
            | _ =>
                if real_core.prim_type_is `u || real_core.union_type_is `u || real_core.nominal_type_is `u
                then format_debug_real `u y
                else $'Rc::<str>::from({ struct W<T>(T); trait D { fn s(&self) -> String; } impl<T: std::fmt::Debug> D for &W<T> { fn s(&self) -> String { format\!("{:?}", self.0) } } trait P { fn s(&self) -> String; } impl<T> P for W<T> { fn s(&self) -> String { String::from("<`t>") } } (&&W(&!y)).s() })' : string
        inl y = real_core.nominal_strip x
        payload `(`y) y
    else $'Rc::<str>::from("<`t>")' : string
and inl format_debug_elements_rust forall el. (x : array_base el) : string =
    join
        inl n = !!!!ArrayLength(`u64, x)
        inl i = mut `u64 (real_core.dyn 0u64)
        inl s = mut `string (real_core.dyn "")
        loopw.while (fun () => !!!!LT(!!!!LayoutIndex(i), n)) fun () =>
            inl j = !!!!LayoutIndex(i)
            inl e = format_debug_element_rust `el !!!!ArrayIndex(x, j)
            s <- (if !!!!EQ(j, 0u64) then e else !!!!LayoutIndex(s) ++#* ", " ++#* e)
            i <- !!!!Add(j, 1u64)
        "[" ++#* !!!!LayoutIndex(s) ++#* "]"
    : string
and inl format_debug_element_rust forall el. (x : el) : string =
    typecase el with
    | ~a * ~b => "(" ++#* format_debug_real `el x ++#* ")"
    | _ => format_debug_real `el x
and inl serialize_union forall t. (x : t) : string =
    join
        $'if type(v0) ~= "table" then return tostring(v0) end' : ()
        real_core.unbox x fun k, v =>
            inl tag : string = $'__v[1].tag'
            typecase `v with
            | () => tag
            | ~a * ~b =>
                typecase a with
                | ~c * ~d =>
                    inl head_str = format_debug_real `c ($'__v[1]._1 ' : c)
                    inl tail_str = format_debug_real `d ($'__v[1]._2  ' : d)
                    inl x = format_debug_real `b ($'__v[1]._3   ' : b)
                    tag ++#* "(" ++#* head_str ++#* ", " ++#* tail_str ++#* ", " ++#* x ++#* ")"
                | _ =>
                    inl head_str = format_debug_real `a ($'__v[1]._1    ' : a)
                    inl tail_str = format_debug_real `b ($'__v[1]._2     ' : b)
                    (++#*) tag ((++#*) "(" ((++#*) head_str ((++#*) ", " ((++#*) tail_str ")"))))
            | ~a =>
                inl arg_str = format_debug_real `a ($'__v[1]._1' : a)
                (++#*) tag ((++#*) "(" ((++#*) arg_str ")"))
    : string
and inl serialize_union_rust forall t. (x : t) : string =
    join
        real_core.unbox x fun k, v =>
            inl name = symbol_to_string_real `(`k)
            typecase `v with
            | () => name
            | _ => (++#*) name ((++#*) "(" ((++#*) (format_debug_real `(`v) v) ")"))
    : string

inl obj_to_string_rust_inner_real forall t. (x : t) : string =
    typecase t with
    | {} => format_debug_real `t x
    | ~a * ~b => format_debug_real `t x
    | ~a -> ~b => format_debug_real `t x
    | array_base _ => format_debug_real `t x
    | ~f ~a => format_debug_real `t x
    | _ =>
        if real_core.prim_type_is `t || real_core.union_type_is `t || real_core.nominal_type_is `t
        then format_debug_real `t x
        else $'Rc::<str>::from(format\!("{}", !x))' : string

inl obj_to_string_rust_real forall t. (x : t) : string =
    typecase t with
    | char => $'Rc::<str>::from((!x as char).encode_utf8(&mut [0u8; 4]) as &str)' : string
    | string => x
    | ~f ~a => format_debug_real `t x
    | _ =>
        if real_core.nominal_type_is `t
        then
            inl x = real_core.nominal_strip x
            obj_to_string_rust_inner_real `(`x) x
        else format_debug_real `t x

### format_debug

In [ ]:
inl format_debug forall t. (x : t) : string =
    real format_debug_real `t x

inl obj_to_string_rust forall t. (x : t) : string =
    real obj_to_string_rust_real `t x
inl obj_to_string_typescript forall t. (x : t) : string =
    if (real typecase t with char => true | _ => false)
    then $'String.fromCharCode(!x)' : string
    else $'String(!x)' : string
inl obj_to_string_wasm forall t. (x : t) : string =
    if (real typecase t with string => true | _ => false) then $'!x ' : string
    elif (real typecase t with bool => true | _ => false) then $'(call $bool_to_str !x)' : string
    elif (real typecase t with char => true | _ => false) then $'(call $char_to_str !x)' : string
    elif (real typecase t with i64 => true | _ => false) then $'(call $i64_to_str !x)' : string
    elif (real typecase t with u64 => true | _ => false) then $'(call $u64_to_str !x)' : string
    elif (real typecase t with i8 => true | i16 => true | i32 => true | _ => false) then $'(call $i64_to_str (i64.extend_i32_s !x))' : string
    elif (real typecase t with u8 => true | u16 => true | u32 => true | _ => false) then $'(call $u64_to_str (i64.extend_i32_u !x))' : string
    else $'backend_switch_has_no_Wasm_arm_in_lib_spiral' : string
inl obj_to_string_zig forall t. (x : t) : string =
    if (real typecase t with string => true | _ => false) then $'!x ' : string
    elif (real typecase t with char => true | _ => false) then $'(std.fmt.allocPrint(spiral_gpa, "{c}", .{ !x }) catch \@panic("out of memory"))' : string
    elif (real typecase t with bool => true | _ => false) then $'(if (!x) "true" else "false")' : string
    elif (real typecase t with f32 => true | f64 => true | i8 => true | i16 => true | i32 => true | i64 => true | u8 => true | u16 => true | u32 => true | u64 => true | _ => false)
    then $'(std.fmt.allocPrint(spiral_gpa, "{d}", .{ !x }) catch \@panic("out of memory"))' : string
    else $'(std.fmt.allocPrint(spiral_gpa, "{any}", .{ !x }) catch \@panic("out of memory"))' : string

### (~#?*)

In [ ]:
inl (~#?*) s =
    s |> format_debug

In [6]:
///- --test --no-agree "tests each backend's native rendering"
///> gleam
///> lua
///> fsharp
///> cuda

#?*("1", "2", "3")
|> _assert_eq (
    backend_switch {
        Gleam = fun () => "#(\"1\", \"2\", \"3\")" : string
        Lua = fun () => "1, 2, 3" : string
        Fsharp = fun () => "struct (\"1\", \"2\", \"3\")" : string
        Python = fun () => "('1', '2', '3')" : string
    }
)

.gleam (gleam)
__assert_eq { expected = #("1", "2", "3") }
.lua (lua)
__assert_eq { expected = 1, 2, 3 }
.fsx (fsharp)
__assert_eq { expected = struct ("1", "2", "3") }
.py (cuda)
__assert_eq { expected = ('1', '2', '3') }

In [7]:
///- --test --no-agree "tests each backend's native rendering"
///> gleam
///> lua
///> fsharp
///> cuda

#?*{ c = "1"; a = "2"; b = "3" }
|> _assert_eq (
    backend_switch {
        Gleam = fun () => "#(\"1\", \"2\", \"3\")" : string
        Lua = fun () => "1, 2, 3" : string
        Fsharp = fun () => "struct (\"1\", \"2\", \"3\")" : string
        Python = fun () => "('1', '2', '3')" : string
    }
)

.gleam (gleam)
__assert_eq { expected = #("1", "2", "3") }
.lua (lua)
__assert_eq { expected = 1, 2, 3 }
.fsx (fsharp)
__assert_eq { expected = struct ("1", "2", "3") }
.py (cuda)
__assert_eq { expected = ('1', '2', '3') }

### unwrap_format

In [ ]:
instance unwrap_format option t = fun x =>
    x
    |> optionm.value
    |> format_debug

instance unwrap_format result t u = fun x =>
    x
    |> resultm.get
    |> format_debug

### obj_to_string

In [ ]:
inl obj_to_string forall t. (x : t) : string =
    if real show_composite_real `t () then real show_real `t x
    else
        backend_switch {
            Gleam = fun () => #?*x : string
            Lua = fun () => #?*x : string
            Fsharp = fun () => $'!x |> _.ToString()' : string
            Python = fun () => $'str(!x)' : string
            Rust = fun () => obj_to_string_rust x
            TypeScript = fun () => obj_to_string_typescript x
            Zig = fun () => obj_to_string_zig x
            Lean = fun () => $'toString !x ' : string
            Wasm = fun () => obj_to_string_wasm x
        }

### (~#?**)

In [ ]:
inl (~#?**) s =
    s |> obj_to_string

### concat_list

In [ ]:
inl concat_list separator input =
    (input, { acc = ""; sep = "" })
    ||> listm.foldBack fun (x : string) { acc sep } => {
        acc = x ++# sep ++# acc
        sep = separator
    }
    |> fun { acc } => acc

In [8]:
///- --test --print-code
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

[
    "1"
    "2"
    "3"
]
|> fun x =>
    inl code = (x : _) |> concat_list "\n"
    code
    |> _assert_eq "1\n2\n3"

.gleam (gleam)
__assert_eq { expected = 1
2
3 }
.fsx (fsharp)
__assert_eq { expected = 1
2
3 }
.py (cuda)
__assert_eq { expected = 1
2
3 }
.rs (rust)
__assert_eq { expected = 1
2
3 }
.ts (typescript)
__assert_eq { expected = 1
2
3 }

0

### slice

In [ ]:
inl slice forall t {number; int}. (from : t) (to : t) s : string =
    backend_switch {
        Gleam = fun () =>
            sm.slice s { from to } : string
        Lua = fun () => sm.slice s { from to } : string
        Fsharp = fun () => sm.slice s { from to } : string
        Python = fun () => sm.slice s { from to } : string
        Rust = fun () => sm.slice s { from to } : string
        TypeScript = fun () => sm.slice s { from to } : string
        Zig = fun () => sm.slice s { from to } : string
        Lean = fun () => sm.slice s { from to } : string
    }

In [9]:
///- --test --print-code
///> fsharp

"abcdef"
|> slice 3i32 3i32
|> _assert_eq "d"

"abcdef"
|> slice 1i32 3i32
|> slice 2i32 2i32
|> _assert_eq "d"

"abcdef"
|> slice 1i32 4i32
|> slice 2i32 3i32
|> slice 1i32 1i32
|> _assert_eq "e"

__assert_eq { expected = d }
__assert_eq { expected = d }
__assert_eq { expected = e }

In [10]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda

"abcdef"
|> slice 1i32 3i32
|> _assert_eq "bcd"

(join "fghij")
|> slice 1i32 3i32
|> _assert_eq "ghi"

"klmno"
|> slice 1i32 (join 3i32)
|> _assert_eq "lmn"

(join "pqrst")
|> slice 1i32 (join 3i32)
|> _assert_eq "qrs"

"uvwxy"
|> slice (join 1i32) 3i32
|> _assert_eq "vwx"

(join "zabcd")
|> slice (join 1i32) 3i32
|> _assert_eq "abc"

"efghi"
|> slice (join 1i32) (join 3i32)
|> _assert_eq "fgh"

(join "efghi")
|> slice (join 1i32) (join 3i32)
|> _assert_eq "fgh"

"abcdef"
|> slice 3i32 4i32
|> _assert_eq "de"

(join "efghi")
|> slice (join 3i32) (join 4i32)
|> _assert_eq "hi"

"bcde"
|> slice (join 0i32) 0i32
|> _assert_eq "b"

.gleam (gleam)
__assert_eq { expected = bcd }
__assert_eq { expected = ghi }
__assert_eq { expected = lmn }
__assert_eq { expected = qrs }
__assert_eq { expected = vwx }
__assert_eq { expected = abc }
__assert_eq { expected = fgh }
__assert_eq { expected = fgh }
__assert_eq { expected = de }
__assert_eq { expected = hi }
__assert_eq { expected = b }
.lua (lua)
__assert_eq { expected = bcd }
__assert_eq { expected = ghi }
__assert_eq { expected = lmn }
__assert_eq { expected = qrs }
__assert_eq { expected = vwx }
__assert_eq { expected = abc }
__assert_eq { expected = fgh }
__assert_eq { expected = fgh }
__assert_eq { expected = de }
__assert_eq { expected = hi }
__assert_eq { expected = b }
.fsx (fsharp)
__assert_eq { expected = bcd }
__assert_eq { expected = ghi }
__assert_eq { expected = lmn }
__assert_eq { expected = qrs }
__assert_eq { expected = vwx }
__assert_eq { expected = abc }
__assert_eq { expected = fgh }
__assert_eq { expected = fgh }
__assert_eq { expected = de }
__assert

### format_pretty_real

In [ ]:
///- --real

inl format_pretty_real forall t. (x : t) : string =
    run_target_args `string `t (fun () => x) function
        | Rust => fun x =>
            inl rec structural forall u. () : bool =
                typecase u with
                | optionm'.option' ~a => if real_core.prim_type_is `a then false else true
                | listm'.list' ~a => if real_core.prim_type_is `a then false else true
                | seq.seq' ~a => if real_core.prim_type_is `a then false else true
                | array_base ~a => if real_core.prim_type_is `a then false else true
                | ~a * ~b => if structural `a () then true else structural `b ()
                | {} => real_core.record_type_fold (fun acc key => forall v. => if acc then true else structural `v ()) false `u
                | _ => real_core.union_type_is `u
            if structural `t ()
            then format_debug_real `t x
            else
                open rust
                inl result = rust.emit_expr `t `std_string_real x ($'"format\!(\\\"{:\#?}\\\", $0)"' : string)
                from_std_string_real result
        | _ => fun _ => format_debug_real `t x

### format_pretty

In [ ]:
inl format_pretty forall t. (x : t) : string =
    real format_pretty_real `t x

### prim

In [ ]:
inl prim x = real
    match x with
    | (x : i8) | (x : i16) | (x : i32) | (x : i64) => "%d", x
    | (x : u8) | (x : u16) | (x : u32) | (x : u64) => "%u", x
    | (x : f32) | (x : f64) => "%f", x
    | (x : string) => "%s", x
    | (x : char) => "%c", x

### printable

In [ ]:
///- --real

prototype printable t : t -> ()

### format_real

In [ ]:
///- --real

let format_real forall t. (x : t) : string =
    inl empty : string =
        backend_switch `string `({}) {
            Gleam =
                (fun () =>
                    ""
                ) : () -> string
            Lua =
                (fun () =>
                    ""
                ) : () -> string
            Fsharp =
                (fun () =>
                    join ""
                ) : () -> string
            Python =
                (fun () =>
                    ""
                ) : () -> string
            Rust =
                (fun () =>
                    ""
                ) : () -> string
            TypeScript =
                (fun () =>
                    ""
                ) : () -> string
            Zig =
                (fun () =>
                    ""
                ) : () -> string
            Lean =
                (fun () =>
                    ""
                ) : () -> string
            C =
                (fun () =>
                    ""
                ) : () -> string
            Delphi =
                (fun () =>
                    ""
                ) : () -> string
        }
    inl result = mut `string empty
    inl rec write x =
        inl p ((a : string), b) =
            inl s : string =
                backend_switch `string `({}) {
                    Gleam =
                        (fun () =>
                            match b with
                            | (_ : string) => b : string
                            | _ => format_debug_real `(`b) b
                        ) : () -> string
                    Lua =
                        (fun () =>
                            match b with
                            | (_ : f32) | (_ : f64) => $'(function(v) if v ~= v then return "NaN" end if v == math.huge then return "Infinity" end if v == -math.huge then return "-Infinity" end for p = 1, 17 do local s = string.format("%." .. p .. "g", v) if tonumber(s) == v then if string.find(s, "[%.e]") then return s end return s .. ".0" end end return tostring(v) end)(!b)' : string
                            | (_ : string) => b : string
                            | _ => format_debug_real `(`b) b
                        ) : () -> string
                    Fsharp =
                        (fun () =>
                            match b with
                            | (_ : f32) | (_ : f64) => $'(let s = (!b).ToString("R", System.Globalization.CultureInfo.InvariantCulture) in if s.Contains(".") || s.Contains("E") || s.Contains("N") || s.Contains("I") then s.Replace("E", "e") else s + ".0")' : string
                            | (_ : string) => b : string
                            | _ => $'$"{!b}"' : string
                        ) : () -> string
                    Python =
                        (fun () =>
                            match b with
                            | (_ : f32) | (_ : f64) => $'(lambda v: (("Infinity" if v == float("inf") else ("-Infinity" if v == float("-inf") else repr(float(v)))) if v == v else "NaN"))(!b)' : string
                            | (_ : string) => b : string
                            | _ => $'f"{!b}"' : string
                        ) : () -> string
                    Rust =
                        (fun () =>
                            match b with
                            | (_ : f32) | (_ : f64) => $'Rc::<str>::from({ let v = !b; if v.is_nan() { String::from("NaN") } else if v.is_infinite() { String::from(if v > 0.0 { "Infinity" } else { "-Infinity" }) } else { format\!("{:?}", v) } })' : string
                            | (_ : string) => b : string
                            | (_ : char) => $'Rc::<str>::from(format\!("{}", !b as char))' : string
                            | _ => $'Rc::<str>::from(format\!("{}", !b))' : string
                        ) : () -> string
                    TypeScript =
                        (fun () =>
                            match b with
                            | (_ : f32) | (_ : f64) => $'(() => { const v = !b; if (Number.isNaN(v)) return "NaN"; if (Number.isFinite(v) === false) return v > 0 ? "Infinity" : "-Infinity"; const s = String(v); return /[.e]/.test(s) ? s : s + ".0"; })()' : string
                            | (_ : string) => b : string
                            | (_ : char) => $'String.fromCharCode(!b)' : string
                            | _ => $'String(!b)' : string
                        ) : () -> string
                    Zig =
                        (fun () =>
                            match b with
                            | (_ : f32) | (_ : f64) => $'(if (std.math.isNan(!b)) "NaN" else if (std.math.isInf(!b)) (if (!b > 0) "Infinity" else "-Infinity") else spiral_shortest: { const s = std.fmt.allocPrint(spiral_gpa, "{d}", .{ !b }) catch \@panic("out of memory"); break :spiral_shortest if (std.mem.indexOfAny(u8, s, ".e") == null) (std.fmt.allocPrint(spiral_gpa, "{s}.0", .{ s }) catch \@panic("out of memory")) else s; })' : string
                            | (_ : string) => b : string
                            | (_ : char) => $'(std.fmt.allocPrint(spiral_gpa, "{c}", .{ !b }) catch \@panic("out of memory"))' : string
                            | (_ : bool) => $'(if (!b) "true" else "false")' : string
                            | _ => $'(std.fmt.allocPrint(spiral_gpa, "{d}", .{ !b }) catch \@panic("out of memory"))' : string
                        ) : () -> string
                    Lean =
                        (fun () =>
                            match b with
                            | (_ : f64) => $'(let v : Float := !b; if v.isNaN then "NaN" else if v.isInf then (if v > 0 then "Infinity" else "-Infinity") else if v == 0 then (if 1 / v < 0 then "-0.0" else "0.0") else Id.run do { let a := Float.abs v; let e : Int := (Float.floor (Float.log10 a)).toInt64.toInt; let mut out := toString a; for p in [1:18] do { let k : Int := e - p + 1; let n := (Float.round (if k >= 0 then a / Float.pow 10 (Float.ofInt k) else a * Float.pow 10 (Float.ofInt (-k)))).toUInt64.toNat; if Float.ofScientific n (k < 0) k.natAbs == a then { let s := toString n; let pos : Int := s.length + k; out := if k >= 0 then s ++ String.ofList (List.replicate k.toNat \'0\') ++ ".0" else if pos > 0 then String.ofList (s.toList.take pos.toNat) ++ "." ++ String.ofList (s.toList.drop pos.toNat) else "0." ++ String.ofList (List.replicate (-pos).toNat \'0\') ++ s; break } }; return (if v < 0 then "-" else "") ++ out })' : string
                            | (_ : f32) => $'(let v : Float := Float32.toFloat !b; if v.isNaN then "NaN" else if v.isInf then (if v > 0 then "Infinity" else "-Infinity") else if v == 0 then (if 1 / v < 0 then "-0.0" else "0.0") else Id.run do { let a := Float.abs v; let e : Int := (Float.floor (Float.log10 a)).toInt64.toInt; let mut out := toString a; for p in [1:18] do { let k : Int := e - p + 1; let n := (Float.round (if k >= 0 then a / Float.pow 10 (Float.ofInt k) else a * Float.pow 10 (Float.ofInt (-k)))).toUInt64.toNat; if (Float.ofScientific n (k < 0) k.natAbs).toFloat32 == a.toFloat32 then { let s := toString n; let pos : Int := s.length + k; out := if k >= 0 then s ++ String.ofList (List.replicate k.toNat \'0\') ++ ".0" else if pos > 0 then String.ofList (s.toList.take pos.toNat) ++ "." ++ String.ofList (s.toList.drop pos.toNat) else "0." ++ String.ofList (List.replicate (-pos).toNat \'0\') ++ s; break } }; return (if v < 0 then "-" else "") ++ out })' : string
                            | (_ : string) => b : string
                            | _ => $'toString !b ' : string
                        ) : () -> string
                    C =
                        (fun () =>
                            match b with
                            | (_ : f64) =>
                                global "#include <stdio.h>"
                                global "#include <stdint.h>"
                                global "#include <stdlib.h>"
                                global "#include <string.h>"
                                global "#include <math.h>"
                                global "static char spiral_format_buffer[64];"
                                global "static inline uint32_t spiral_format_shortest(double x, int single) { int n = 0; if (isnan(x)) n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"NaN\"); else if (isinf(x)) n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, x > 0 ? \"Infinity\" : \"-Infinity\"); else { for (int p = 1; p <= 17; p++) { n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"%.*g\", p, x); if (single ? strtof(spiral_format_buffer, NULL) == (float)x : strtod(spiral_format_buffer, NULL) == x) break; } if (!strpbrk(spiral_format_buffer, \".e\")) { spiral_format_buffer[n++] = '.'; spiral_format_buffer[n++] = '0'; spiral_format_buffer[n] = 0; } } return (uint32_t)n + 1u; }"
                                $'StringLit(spiral_format_shortest((double)(!b), 0), spiral_format_buffer)' : string
                            | (_ : f32) =>
                                global "#include <stdio.h>"
                                global "#include <stdint.h>"
                                global "#include <stdlib.h>"
                                global "#include <string.h>"
                                global "#include <math.h>"
                                global "static char spiral_format_buffer[64];"
                                global "static inline uint32_t spiral_format_shortest(double x, int single) { int n = 0; if (isnan(x)) n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"NaN\"); else if (isinf(x)) n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, x > 0 ? \"Infinity\" : \"-Infinity\"); else { for (int p = 1; p <= 17; p++) { n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"%.*g\", p, x); if (single ? strtof(spiral_format_buffer, NULL) == (float)x : strtod(spiral_format_buffer, NULL) == x) break; } if (!strpbrk(spiral_format_buffer, \".e\")) { spiral_format_buffer[n++] = '.'; spiral_format_buffer[n++] = '0'; spiral_format_buffer[n] = 0; } } return (uint32_t)n + 1u; }"
                                $'StringLit(spiral_format_shortest((double)(!b), 1), spiral_format_buffer)' : string
                            | (_ : string) => b : string
                            | (_ : char) =>
                                global "#include <stdio.h>"
                                global "#include <stdint.h>"
                                global "#include <stdlib.h>"
                                global "#include <string.h>"
                                global "static char spiral_format_buffer[64];"
                                global "static inline uint32_t spiral_format_plain_char(int32_t x) { return (uint32_t)snprintf(spiral_format_buffer, sizeof spiral_format_buffer, \"%c\", (int)x) + 1u; }"
                                $'StringLit(spiral_format_plain_char(!b), spiral_format_buffer)' : string
                            | _ => format_debug_real `(`b) b
                        ) : () -> string
                    Delphi =
                        (fun () =>
                            match b with
                            | (_ : f64) =>
                                global "function SpiralFormatFloat(x: Double; single: Boolean): AnsiString;\nvar p: Integer; s: string; fs: TFormatSettings; d: Double; f: Single;\nbegin\n  if IsNan(x) then Exit('NaN');\n  if IsInfinite(x) then begin if x > 0 then Exit('Infinity') else Exit('-Infinity'); end;\n  fs := FormatSettings; fs.DecimalSeparator := '.';\n  s := '';\n  for p := 1 to 17 do begin\n    s := FloatToStrF(x, ffGeneral, p, 0, fs);\n    if single then begin f := StrToFloat(s, fs); if f = x then Break; end\n    else begin d := StrToFloat(s, fs); if d = x then Break; end;\n  end;\n  if (Pos('.', s) = 0) and (Pos('E', s) = 0) and (Pos('e', s) = 0) then s := s + '.0';\n  Result := StringReplace(s, 'E', 'e', []);\nend;"
                                $'SpiralFormatFloat(!b, False)' : string
                            | (_ : f32) =>
                                global "function SpiralFormatFloat(x: Double; single: Boolean): AnsiString;\nvar p: Integer; s: string; fs: TFormatSettings; d: Double; f: Single;\nbegin\n  if IsNan(x) then Exit('NaN');\n  if IsInfinite(x) then begin if x > 0 then Exit('Infinity') else Exit('-Infinity'); end;\n  fs := FormatSettings; fs.DecimalSeparator := '.';\n  s := '';\n  for p := 1 to 17 do begin\n    s := FloatToStrF(x, ffGeneral, p, 0, fs);\n    if single then begin f := StrToFloat(s, fs); if f = x then Break; end\n    else begin d := StrToFloat(s, fs); if d = x then Break; end;\n  end;\n  if (Pos('.', s) = 0) and (Pos('E', s) = 0) and (Pos('e', s) = 0) then s := s + '.0';\n  Result := StringReplace(s, 'E', 'e', []);\nend;"
                                $'SpiralFormatFloat(!b, True)' : string
                            | (_ : string) => b : string
                            | (_ : char) => $'string(!b)' : string
                            | _ => format_debug_real `(`b) b
                        ) : () -> string
                }
            backend_switch `(()) `({}) {
                Gleam =
                    (fun () =>
                        exec_unit ((fun () => result <- append_real s ((~*) `string result)) : () -> ())
                    ) : () -> ()
                Lua =
                    (fun () =>
                        exec_unit ((fun () => result <- append_real s ((~*) `string result)) : () -> ())
                    ) : () -> ()
                Fsharp =
                    (fun () =>
                        join
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Python =
                    (fun () =>
                        join
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Rust =
                    (fun () =>
                        join
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                TypeScript =
                    (fun () =>
                        join
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Zig =
                    (fun () =>
                        join
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Lean =
                    (fun () =>
                        join
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()

                C =
                    (fun () =>
                        join
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Delphi =
                    (fun () =>
                        join
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
            }

        match x with
        | () => ()
        | (x : i8) | (x : i16) | (x : i32) | (x : i64) => p ("%d", x)
        | (x : u8) | (x : u16) | (x : u32) | (x : u64) => p ("%u", x)
        | (x : f32) | (x : f64) => p ("%f", x)
        | (x : string) => p ("%s", x)
        | (x : char) => p ("%c", x)
        | (x : bool) => p ("%s", if x then "true" else "false")
        | (a,b) => write a . write ", " . write b
        | {} as x =>
            write "{ "
            inl _result =
                real_core.record_fold
                    fun { state = separator key value } =>
                        write separator
                        write (symbol_to_string_real `(`key)) . write " = " . write value
                        "; "
                    () x
            write " }"
        | x when real_core.symbol_is x => write (symbol_to_string_real `(`x))
        | x when real_core.function_is x => write (x ())
        | x when real_core.union_is x =>
            if real_core.prototype_has `(`x) printable then printable `(`x) x
            else
                inl temp () =
                    real_core.unbox x (fun (k, v) =>
                        write k
                        match v with
                        | () => ()
                        | _ => (join write "(" . write v . write ")") : ()
                        )
                write (format_debug_real `(`x) x)
        | x when real_core.nominal_is x =>
            if real_core.prototype_has `(`x) printable then printable `(`x) x
            else write (format_pretty_real `(`x) x)
        | x => write (format_debug_real `(`x) x)
    write x
    (~*) `string result

### format

In [ ]:
inl format forall t. (x : t) : string =
    real show_real `t x

In [11]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript
///> c

("1", "2", ["3"; "4"], { b = "5"; c = "6"; a = fun () => "7" }, ;[ 8i32 ])
|> format
|> _assert_eq "(1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8])"

.gleam (gleam)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.lua (lua)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.fsx (fsharp)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.py (cuda)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.rs (rust)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.ts (typescript)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.c (c)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }

0

### (~#?)

In [ ]:
inl (~#?) s =
    s |> format

### (++#?)

In [ ]:
inl (++#?) (a : string) (b : _) : string =
    a ++# #?b

### show_real

In [ ]:
///- --real

inl rec show_real forall t. (x : t) : string =
    typecase t with
    | string => x
    | ~a * ~b => "(" ++#* show_items_real `t x ++#* ")"
    | {} =>
        inl _, fields =
            real_core.record_fold
                fun { state = separator, acc key value } =>
                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* show_real `(`value) value
                ("", "") x
        "{ " ++#* fields ++#* " }"
    | array_base ~el => show_elements_real `el x
    | _ =>
        if real_core.union_type_is `t
        then show_union_real `t x
        elif real_core.nominal_type_is `t
        then show_nominal_real `t x
        else format_real `t x
and inl show_items_real forall t. (x : t) : string =
    typecase t with
    | () * ~b =>
        inl _, b = x
        show_items_real `b b
    | ~a * ~b =>
        inl a, b = x
        show_real `a a ++#* ", " ++#* show_items_real `b b
    | _ => show_real `t x
and inl show_union_real forall t. (x : t) : string =
    join
        real_core.unbox x fun k, v =>
            inl name = symbol_to_string_real `(`k)
            typecase `v with
            | () => name
            | _ => name ++#* "(" ++#* show_items_real `(`v) v ++#* ")"
    : string
and inl show_nominal_real forall t. (x : t) : string =
    inl y = real_core.nominal_strip x
    if real_core.prototype_has `t show_as
    then show_as `t x
    elif real_core.prototype_has `t printable
    then format_real `t x
    elif show_structural_real `(`y) ()
    then show_real `(`y) y
    else format_debug_real `t x
and inl show_structural_real forall u. () : bool =
    typecase u with
    | string => true
    | ~a * ~b => true
    | {} => true
    | array_base _ => true
    | _ => real_core.prim_type_is `u || real_core.union_type_is `u || real_core.nominal_type_is `u
and inl show_elements_real forall el. (x : array_base el) : string =
    join
        inl n = !!!!ArrayLength(`u64, x)
        inl i = mut `u64 (real_core.dyn 0u64)
        inl s = mut `string (real_core.dyn "")
        loopw.while (fun () => !!!!LT(!!!!LayoutIndex(i), n)) fun () =>
            inl j = !!!!LayoutIndex(i)
            inl e = show_real `el !!!!ArrayIndex(x, j)
            s <- (if !!!!EQ(j, 0u64) then e else !!!!LayoutIndex(s) ++#* ", " ++#* e)
            i <- !!!!Add(j, 1u64)
        "[" ++#* !!!!LayoutIndex(s) ++#* "]"
    : string

inl show_composite_real forall t. () : bool =
    typecase t with
    | string => false
    | ~a * ~b => true
    | {} => true
    | array_base _ => true
    | _ => real_core.union_type_is `t

### show

In [ ]:
inl show forall t. (x : t) : string =
    real show_real `t x

In [12]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

show ("1", 2i32, ["3"; "4"], { b = "5"; c = (6i32, Some 'x') }, ;[ ("7", 8i32) ])
|> _assert_eq "(1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)])"

.gleam (gleam)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.lua (lua)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.fsx (fsharp)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.py (cuda)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.rs (rust)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.ts (typescript)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }

0

#### show_floats (test)

In [13]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript
///> c
///> zig
///> delphi
///> lean

show (1.5f64, 0.1f64, 2.0f64, 1.5707963267948966f64, -0.25f64)
|> _assert_eq "(1.5, 0.1, 2.0, 1.5707963267948966, -0.25)"

.gleam (gleam)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.lua (lua)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.fsx (fsharp)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.py (cuda)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.rs (rust)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.ts (typescript)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.c (c)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
zig: unavailable on this machine (`zig` was not found)
delphi: unavailable on this machine (`dcc32` was not found (nor fpc))
lean: unavailable on this machine (`lean` was not found)

0

### to_char_array

In [ ]:
inl to_char_array (str : string) : array_base char =
    am.init (str |> length) (fun i => str |> index i)
    |> fun (a x : _ int _) => x

In [14]:
///- --test
///> fsharp
///> cuda

"abc"
|> to_char_array
|> format
|> _assert_eq #?(;[ 'a'; 'b'; 'c' ])

.fsx (fsharp)
__assert_eq { expected = [a, b, c] }
.py (cuda)
__assert_eq { expected = [a, b, c] }

### concat_array

In [ ]:
inl concat_array (separator : string) (input : a int string) =
    (input, { acc = ""; sep = "" })
    ||> am.foldBack fun (x : string) { acc sep } =>
            { acc = x ++# sep ++# acc; sep = separator }
    |> fun { acc } => acc

In [15]:
///- --test --print-code
///> fsharp
///> rust
///> typescript
///> cuda

;[
    "1"
    "2"
    "3"
]
|> fun x =>
    inl code = (a x : _ int _) |> concat_array "\n"
    code
    |> _assert_eq "1\n2\n3"

.fsx (fsharp)
__assert_eq { expected = 1
2
3 }
.rs (rust)
__assert_eq { expected = 1
2
3 }
.ts (typescript)
__assert_eq { expected = 1
2
3 }
.py (cuda)
__assert_eq { expected = 1
2
3 }

0

### ellipsis

In [ ]:
inl ellipsis (max : i32) (s : string) =
    if sm.length s <= max
    then s
    else s |> slice 0 (max - 1) |> append "..."

In [16]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"12345"
|> ellipsis 2
|> _assert_eq "12..."

"12345"
|> ellipsis 4
|> _assert_eq "1234..."

.gleam (gleam)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }
.fsx (fsharp)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }
.py (cuda)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }
.rs (rust)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }
.ts (typescript)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }

0

## fsharp

### last_index_of

In [ ]:
inl last_index_of (search : string) (s : string) : i32 =
    inl search = dyn search
    run_target function
        | Rust => fun () => $'!s.rfind(&*!search).map(|i| i as i32).unwrap_or(-1)' : i32
        | _ => fun () => $'!s.LastIndexOf !search ' : i32

### index_of_char

In [ ]:
inl index_of_char (search : char) (s : string) : int =
    $'!s.IndexOf !search '

### index_of_char_from

In [ ]:
inl index_of_char_from (from : i32) (c : char) (s : string) : i32 =
    inl len = s |> length
    inl rec body i =
        if i >= len
        then -1i32
        elif c = (s |> index i)
        then i - from
        else 루프 (i + 1i32)
    and inl 루프 i = join_body_unit body (if var_is i then i else len) i
    from |> 루프

In [17]:
///- --test

"abcdefabc"
|> index_of_char_from 2i32 'a'
|> _assert_eq 4i32

__assert_eq { expected = 4 }

0

### index_of

In [ ]:
inl index_of (search : string) (s : string) : int =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/list"
            global "import gleam/option"
            global "import gleam/string"
            inl g : list string =
                $'!s |> string.to_graphemes'
            $'let search = !search ' : ()
            $'let x = !g |> list.index_fold (option.None, fn(acc, item, index) {' : ()
            $'  case acc {' : ()
            $'    option.None if item == search -> option.Some(index)' : ()
            $'    _ -> acc' : ()
            $'  }' : ()
            $'})' : ()
            $'x |> option.unwrap(-1)' : int
        Lua = fun () => $'string.find(!s, !search) - 1' : int
        Fsharp = fun () => $'!s.IndexOf !search ' : int
        Python = fun () => $'!s.find(!search)' : int
        Rust = fun () => $'!s.find(&*!search).map(|i| i as `int).unwrap_or(-1)' : int
        TypeScript = fun () => $'!s.indexOf(!search)' : int
        Zig = fun () => $'(if (std.mem.indexOf(u8, !s, !search)) |i| \@as(`int, \@intCast(i)) else -1)' : int
        Lean = fun () => $'(if (!search).isEmpty then 0 else match (!s).splitOn !search with | [_] => -1 | p :: _ => `int.ofNat p.utf8ByteSize | [] => -1)' : int
    }

In [18]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"123"
|> index_of "2"
|> _assert_eq 1

.gleam (gleam)
__assert_eq { expected = 1 }
.fsx (fsharp)
__assert_eq { expected = 1 }
.py (cuda)
__assert_eq { expected = 1 }
.rs (rust)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }

0

### replicate

In [ ]:
inl replicate (n : int) (s : string) : string =
    inl rec body i acc =
        if i >= n
        then acc
        else 루프 (i + 1) (acc ++# s)
    and inl 루프 i = join_body_unit body n i
    루프 0 ""

In [19]:
///- --test
///> gleam
///> fsharp

"12"
|> replicate (join 3)
|> _assert_eq "121212"

.gleam (gleam)
__assert_eq { expected = 121212 }
.fsx (fsharp)
__assert_eq { expected = 121212 }

In [20]:
///- --test --print-code
///> gleam
///> fsharp

"12"
|> replicate 3
|> _assert_eq "121212"

.gleam (gleam)
__assert_eq { expected = 121212 }
.fsx (fsharp)
__assert_eq { expected = 121212 }

In [21]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"12"
|> replicate 3
|> _assert_eq "121212"

.gleam (gleam)
__assert_eq { expected = 121212 }
.fsx (fsharp)
__assert_eq { expected = 121212 }
.py (cuda)
__assert_eq { expected = 121212 }
.rs (rust)
__assert_eq { expected = 121212 }
.ts (typescript)
__assert_eq { expected = 121212 }

0

### pad_left

In [ ]:
inl pad_left (total_width : int) (padding_char : char) (s : string) : string =
    inl padding = padding_char |> obj_to_string |> replicate (total_width - length s)
    padding ++# s

### pad_right

In [ ]:
inl pad_right (total_width : int) (padding_char : char) (s : string) : string =
    inl padding = padding_char |> obj_to_string |> replicate (total_width - length s)
    s ++# padding

In [22]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"123"
|> pad_right 6 ' '
|> _assert_eq "123   "

.gleam (gleam)
__assert_eq { expected = 123    }
.fsx (fsharp)
__assert_eq { expected = 123    }
.py (cuda)
__assert_eq { expected = 123    }
.rs (rust)
__assert_eq { expected = 123    }
.ts (typescript)
__assert_eq { expected = 123    }

0

### convert_to_utf32

In [ ]:
inl convert_to_utf32 (c : char) : int =
    backend_switch {
        Gleam = fun () => c |> convert.i32
        Lua = fun () => c |> convert.i32
        Fsharp = fun () =>
            run_target_args' c function
            | Fsharp => fun c => $'System.Char.ConvertToUtf32 (string !c, 0)' : int
            | _ => fun c => c |> convert.i32
        Python = fun () => $'ord(!c)' : int
        Rust = fun () => $'(!c as `int)' : int
        TypeScript = fun () => $'(!c)' : int
    }

### ends_with

In [ ]:
inl ends_with (value : string) (s : string) : bool =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!s |> string.ends_with(!value)' : bool
        Fsharp = fun () => $'!s.EndsWith (!value, false, null)' : bool
        Python = fun () => $'!s.endswith(!value)' : bool
        Rust = fun () => $'!s.ends_with(&*!value)' : bool
        TypeScript = fun () => $'!s.endsWith(!value)' : bool
        Lua = fun () => $'!value == "" or !s:sub(-##!value) == !value ' : bool
        Zig = fun () => $'std.mem.endsWith(u8, !s, !value)' : bool
        Lean = fun () => $'(!s).endsWith !value ' : bool
    }

### starts_with

In [ ]:
inl starts_with (value : string) (s : string) : bool =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!s |> string.starts_with(!value)' : bool
        Lua = fun () => $'!value == "" or !s:sub(1, ##!value) == !value ' : bool
        Fsharp = fun () => $'!s.StartsWith (!value, false, null)' : bool
        Python = fun () => $'!s.startswith(!value)' : bool
        Rust = fun () => $'!s.starts_with(&*!value)' : bool
        TypeScript = fun () => $'!s.startsWith(!value)' : bool
        Zig = fun () => $'std.mem.startsWith(u8, !s, !value)' : bool
        Lean = fun () => $'(!s).startsWith !value ' : bool
    }

### is_white_space

In [ ]:
inl is_white_space (c : char) : bool =
    c |> $'System.Char.IsWhiteSpace'

### substring

In [ ]:
inl substring (start : i32) (len : i32) (str : string) : string =
    if is_rust ()
    then $'Rc::<str>::from(&(!str)[((!start) as usize)..(((!start) + (!len)) as usize)])'
    else $'!str.Substring (!start, !len)'

### to_lower

In [ ]:
inl to_lower (input : string) : string =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!input |> string.lowercase' : string
        Lua = fun () => $'string.lower(!input)' : string
        Fsharp = fun () => $'!input.ToLower' () : string
        Python = fun () => $'!input.lower()' : string
        Rust = fun () => $'Rc::<str>::from(!input.to_lowercase())' : string
        TypeScript = fun () => $'!input.toLowerCase()' : string
        Zig = fun () => $'(std.ascii.allocLowerString(spiral_gpa, !input) catch \@panic("out of memory"))' : string
        Lean = fun () => $'(!input).toLower' : string
    }

### to_upper

In [ ]:
inl to_upper (input : string) : string =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!input |> string.uppercase' : string
        Lua = fun () => $'string.upper(!input)' : string
        Fsharp = fun () => $'!input.ToUpper' () : string
        Python = fun () => $'!input.upper()' : string
        Rust = fun () => $'Rc::<str>::from(!input.to_uppercase())' : string
        TypeScript = fun () => $'!input.toUpperCase()' : string
        Zig = fun () => $'(std.ascii.allocUpperString(spiral_gpa, !input) catch \@panic("out of memory"))' : string
        Lean = fun () => $'(!input).toUpper' : string
    }

### char_to_upper

In [ ]:
inl char_to_upper (input : char) : char =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!input |> string.uppercase' : char
        Lua = fun () => $'string.upper(!input)' : char
        Fsharp = fun () => $'System.Char.ToUpper !input ' : char
        Python = fun () => $'!input.upper()' : char
        Rust = fun () => $'!input.to_ascii_uppercase()' : char
        TypeScript = fun () => $'String.fromCharCode(!input).toUpperCase().charCodeAt(0)' : char
        Zig = fun () => $'std.ascii.toUpper(!input)' : char
        Lean = fun () => $'(!input).toUpper' : char
    }

### builder_append

In [ ]:
inl builder_append (item : string) (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = fun () =>
            $'!sb <> !item ' : string_builder
        Lua = fun () =>
            $'!sb .. !item ' : string_builder
        Fsharp = fun () =>
            ($'!sb.Append' item : string_builder) |> ignore
            sb
        Python = fun () =>
            ($'!sb.write(!item)' : int) |> ignore
            sb
        Rust = fun () => $'{ let mut r = !sb.clone(); r.push_str(&*!item); r }' : string_builder
        TypeScript = fun () => $'!sb + !item ' : string_builder
    }

In [23]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"ab"
|> string_builder
|> builder_append "cd"
|> builder_append "ef"
|> obj_to_string
|> _assert_eq "abcdef"

.gleam (gleam)
__assert_eq { expected = abcdef }
.fsx (fsharp)
__assert_eq { expected = abcdef }
.py (cuda)
__assert_eq { expected = abcdef }
.rs (rust)
__assert_eq { expected = abcdef }
.ts (typescript)
__assert_eq { expected = abcdef }

0

### builder_append_line

In [ ]:
inl builder_append_line (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = fun () =>
            sb |> builder_append "\n"
        Lua = fun () =>
            sb |> builder_append "\n"
        Fsharp = fun () =>
            ($'!sb.AppendLine ()' : string_builder) |> ignore
            sb
        Python = fun () =>
            sb |> builder_append "\n"
        Rust = fun () =>
            sb |> builder_append "\n"
        TypeScript = fun () =>
            sb |> builder_append "\n"
    }

### builder_replace

In [ ]:
inl builder_replace (old : string) (new : string) (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!sb |> string.replace(!old, !new)' : string_builder
        Lua = fun () =>
            $'string.gsub(!sb, !old, !new)' : string_builder
        Fsharp = fun () =>
            ($'!sb.Replace (!old, !new)' : string_builder) |> ignore
            sb
        Python = fun () =>
            ($'!sb.getvalue().replace(!old, !new)' : string) |> string_builder
        Rust = fun () => $'!sb.replace(&*!old, &*!new)' : string_builder
        TypeScript = fun () => $'!sb.split(!old).join(!new)' : string_builder
    }

In [24]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"ab"
|> string_builder
|> builder_append "cd"
|> builder_replace "cd" "ef"
|> builder_append "gh"
|> obj_to_string
|> _assert_eq "abefgh"

.gleam (gleam)
__assert_eq { expected = abefgh }
.fsx (fsharp)
__assert_eq { expected = abefgh }
.py (cuda)
__assert_eq { expected = abefgh }
.rs (rust)
__assert_eq { expected = abefgh }
.ts (typescript)
__assert_eq { expected = abefgh }

0

### builder_clear

In [ ]:
inl builder_clear (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = fun () =>
            $'let !sb = ""'
            sb
        Lua = fun () =>
            $'!sb = ""'
            sb
        Fsharp = fun () =>
            ($'!sb.Clear' () : string_builder) |> ignore
            sb
        Python = fun () =>
            ($'!sb.truncate(0)' : int) |> ignore
            ($'!sb.seek(0)' : int) |> ignore
            sb
        Rust = fun () => $'String::new()' : string_builder
        TypeScript = fun () => $'""' : string_builder
    }

In [25]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"ab"
|> string_builder
|> builder_append "cd"
|> builder_clear
|> builder_append "gh"
|> obj_to_string
|> _assert_eq "gh"

.gleam (gleam)
__assert_eq { expected = gh }
.fsx (fsharp)
__assert_eq { expected = gh }
.py (cuda)
__assert_eq { expected = gh }
.rs (rust)
__assert_eq { expected = gh }
.ts (typescript)
__assert_eq { expected = gh }

0

### builder_insert

In [ ]:
inl builder_insert (i : int) (s : string) (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = fun () =>
            inl sb = #?sb
            inl a = sb |> slice 0 (i - 1)
            inl b = sb |> slice i (length sb - 1)
            [ a; s; b ] |> concat_list "" |> string_builder
        Lua = fun () =>
            inl sb = #?sb
            inl a = sb |> slice 0 (i - 1)
            inl b = sb |> slice i (length sb - 1)
            [ a; s; b ] |> concat_list "" |> string_builder
        Fsharp = fun () =>
            ($'!sb.Insert (!i, !s)' : string_builder) |> ignore
            sb
        Python = fun () =>
            inl sb = sb |> obj_to_string
            $'"".join([!sb[:!i], !s, !sb[!i:]])' |> string_builder
        Rust = fun () => $'{ let mut r = !sb.clone(); let at = r.char_indices().nth(!i as usize).map(|(b, _)| b).unwrap_or(r.len()); r.insert_str(at, &*!s); r }' : string_builder
    }

In [26]:
///- --test
///> gleam
///> fsharp
///> cuda

"ab"
|> string_builder
|> builder_insert 1 "gh"
|> obj_to_string
|> _assert_eq "aghb"

.gleam (gleam)
__assert_eq { expected = aghb }
.fsx (fsharp)
__assert_eq { expected = aghb }
.py (cuda)
__assert_eq { expected = aghb }

### span_from

In [ ]:
inl span_from pred str i0 =
    inl len = str |> length
    inl rec body (i : int) =
        if i >= len
        then i
        elif str |> index i |> pred
        then 루프 (i + 1)
        else i
    and inl 루프 i = join_body_unit body (if var_is i then i else len) i
    i0 |> 루프

### char_contains

In [ ]:
inl char_contains (chars : list char) (c : char) : bool =
    chars |> listm'.exists' ((=) c)

### trim_chars

In [ ]:
inl trim_chars () =
    [ ' '; '\t'; '\r'; '\n' ]

### trim_start

In [ ]:
inl trim_start (chars : list char) (input : string) : string =
    inl chars = if chars = [] then trim_chars () else chars
    inl len = input |> length
    let rec 루프 (i : int) =
        if i >= len
        then len
        elif input |> index i |> char_contains chars
        then 루프 (i + 1)
        else i
    input |> slice (루프 0) (len - 1)

In [27]:
///- --test
///> gleam
///> fsharp
///> cuda

"  abc  "
|> trim_start []
|> _assert_eq "abc  "

"abcde"
|> trim_start []
|> _assert_eq "abcde"

"abcde"
|> trim_start [ 'a' ]
|> _assert_eq "bcde"

"abcde"
|> trim_start [ 'b'; 'a' ]
|> _assert_eq "cde"

"abcde"
|> trim_start [ 'c'; 'b'; 'a' ]
|> _assert_eq "de"

.gleam (gleam)
__assert_eq { expected = abc   }
__assert_eq { expected = abcde }
__assert_eq { expected = bcde }
__assert_eq { expected = cde }
__assert_eq { expected = de }
.fsx (fsharp)
__assert_eq { expected = abc   }
__assert_eq { expected = abcde }
__assert_eq { expected = bcde }
__assert_eq { expected = cde }
__assert_eq { expected = de }
.py (cuda)
__assert_eq { expected = abc   }
__assert_eq { expected = abcde }
__assert_eq { expected = bcde }
__assert_eq { expected = cde }
__assert_eq { expected = de }

### trim_end

In [ ]:
inl trim_end (chars : list char) (input : string) : string =
    inl chars = if chars = [] then trim_chars () else chars
    inl len = input |> length
    let rec 루프 (i : int) =
        if i <= 0
        then -1
        elif input |> index (i - 1) |> char_contains chars
        then 루프 (i - 1)
        else i - 1
    input |> slice 0 (루프 len)

In [28]:
///- --test
///> gleam
///> fsharp
///> cuda

"  def  "
|> trim_end []
|> _assert_eq "  def"

"abcde"
|> trim_end []
|> _assert_eq "abcde"

"abcde"
|> trim_end [ 'e' ]
|> _assert_eq "abcd"

"abcde"
|> trim_end [ 'e'; 'd' ]
|> _assert_eq "abc"

"abcde"
|> trim_end [ 'e'; 'd'; 'c' ]
|> _assert_eq "ab"

.gleam (gleam)
__assert_eq { expected =   def }
__assert_eq { expected = abcde }
__assert_eq { expected = abcd }
__assert_eq { expected = abc }
__assert_eq { expected = ab }
.fsx (fsharp)
__assert_eq { expected =   def }
__assert_eq { expected = abcde }
__assert_eq { expected = abcd }
__assert_eq { expected = abc }
__assert_eq { expected = ab }
.py (cuda)
__assert_eq { expected =   def }
__assert_eq { expected = abcde }
__assert_eq { expected = abcd }
__assert_eq { expected = abc }
__assert_eq { expected = ab }

### trim_with

In [ ]:
inl trim_with (chars : list char) (input : string) : string =
    input
    |> trim_start chars
    |> trim_end chars

In [29]:
///- --test

"  jkl  "
|> trim_with [ ' '; '\t'; '\n' ]
|> _assert_eq "jkl"

__assert_eq { expected = jkl }

0

### trim

In [ ]:
inl trim (input : string) : string =
    input |> trim_with []

In [30]:
///- --test

"  ghi  "
|> trim
|> _assert_eq "ghi"

__assert_eq { expected = ghi }

0

### concat

In [ ]:
inl concat (a : string) (b : seq.seq' string) : string =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!b |> string.join(!a)' : string
        Lua = fun () => $'table.concat(!b, !a)' : string
        Fsharp = fun () =>
            inl a =
                if a = "\n"
                then join a
                else a
            b |> $'String.concat' a : string
        Python = fun () =>
            $'!a.join(!b)' : string
        Rust = fun () => $'Rc::<str>::from(!b.iter().map(|x| x.to_string()).collect::<Vec<_>>().join(&*!a))' : string
        TypeScript = fun () => $'!b.join(!a)' : string
    }

### length'

In [ ]:
inl length' forall dim. (input : string) : dim =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'string.length(!input)' : dim
        Lua = fun () => $'##!input ' : dim
        Fsharp = fun () => input |> $'String.length' : dim
        Python = fun () => $'len(!input)' : dim
        Rust = fun () => $'(!input.len() as `dim)' : dim
        Zig = fun () => $'\@as(`dim, \@intCast(!input.len))' : dim
        Lean = fun () => $'(`dim.ofNat (!input).utf8ByteSize)' : dim
    }

In [31]:
///- --test
///> fsharp
///> cuda

"abc"
|> length'
|> _assert_eq 3i32

.fsx (fsharp)
__assert_eq { expected = 3 }
.py (cuda)
__assert_eq { expected = 3 }

### to_string any

In [ ]:
instance to_string any =
    obj_to_string

### replace

In [ ]:
inl replace (old : string) (new : string) (s : string) : string =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!s |> string.replace(!old, !new)' : string
        Lua = fun () =>
            $'string.gsub(!s, !old, !new)' : string
        Fsharp = fun () =>
            $'!s.Replace (!old, !new)' : string
        Python = fun () =>
            $'!s.replace(!old, !new)' : string
        Rust = fun () =>
            $'Rc::<str>::from(!s.replace(&*!old, &*!new))' : string
        TypeScript = fun () =>
            $'!s.split(!old).join(!new)' : string
        Zig = fun () =>
            $'(std.mem.replaceOwned(u8, spiral_gpa, !s, !old, !new) catch \@panic("out of memory"))' : string
        Lean = fun () =>
            $'(!s).replace !old !new ' : string
    }

### split

In [ ]:
inl split (separator : string) (str : string) : array_base string =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            ($'!str |> string.split(!separator)' : listm'.list' string)
            |> listm'.to_array'
        Fsharp = fun () => $'!str.Split !separator ' : array_base string
        Python = fun () => $'!str.split(!separator)' : array_base string
        Rust = fun () =>
            $'Rc::new(RefCell::new(!str.split(&*!separator).map(|x| Rc::<str>::from(x)).collect::<Vec<Rc<str>>>()))'
            : array_base string
        TypeScript = fun () => $'!str.split(!separator)' : array_base string
    }

### split_string

In [ ]:
inl split_string (separator : array_base string) (str : string) : array_base string =
    run_target_args (fun () => str, separator) function
        | Fsharp => fun str, separator => $'!str.Split (!separator, System.StringSplitOptions.None)'
        | _ => fun str, separator => str |> split ((a separator : _ int _) |> seq.of_array |> concat (join ""))

### join'

In [ ]:
inl join' (concat : string) (s : a int string) : string =
    $'System.String.Join (!concat, !s)'

### encoding

In [ ]:
nominal encoding = $'System.Text.Encoding'

### encoding_utf8

In [ ]:
inl encoding_utf8 () : encoding =
    $'`encoding.UTF8'

### utf8_get_bytes

In [ ]:
inl utf8_get_bytes (s : string) : a i32 u8 =
    run_target function
        | Rust => fun () => $'Rc::new(RefCell::new(!s.as_bytes().to_vec()))' : a i32 u8
        | _ => fun () => s |> (encoding_utf8 () |> $'_.GetBytes') : a i32 u8

### byte_to_string

In [ ]:
inl byte_to_string (format : string) (x : u8) : string =
    $'!x.ToString' format

## rust

### str

In [ ]:
nominal str =
    `(
        rust_type
            `($'Str')
            `($'str')
            "type Str = string"
    )

### chars

In [ ]:
inl chars (x : rust.ref str) : rust.mut' (into_iterator char) =
    !\\(x, $'$"$0.chars()"')

### char_is_alphanumeric

In [ ]:
inl char_is_alphanumeric (x : char) : bool =
    !\\(x, $'$"$0.is_alphanumeric()"')

### byte_slice

In [ ]:
inl byte_slice (s : string) : rust.ref (am'.slice u8) =
    backend_switch {
        Rust = fun () => $'{ let owned: Rc<str> = (!s).clone(); Box::leak(owned.as_bytes().to_vec().into_boxed_slice()) }' : rust.ref (am'.slice u8)
    }

### display

In [ ]:
nominal display t =
    `(
        rust_type
            `($'std_fmt_Display<`t>')
            `($'std::fmt::Display<`t>')
            "type std_fmt_Display<'T> = class end"
    )

### base64_decode_error

In [ ]:
nominal base64_decode_error =
    `(
        rust_type
            `($'base64_DecodeError')
            `($'base64::DecodeError')
            "type base64_DecodeError = class end"
    )

### borsh_io_error

In [ ]:
nominal borsh_io_error =
    `(
        rust_type
            `($'borsh_io_Error')
            `($'borsh::io::Error')
            "type borsh_io_Error = class end"
    )

### utf8_error

In [ ]:
nominal utf8_error =
    `(
        rust_type
            `($'std_str_Utf8Error')
            `($'std::str::Utf8Error')
            "type std_str_Utf8Error = class end"
    )

### from_utf8_error

In [ ]:
nominal from_utf8_error =
    `(
        rust_type
            `($'std_string_FromUtf8Error')
            `($'std::string::FromUtf8Error')
            "type std_string_FromUtf8Error = class end"
    )

### json_value

In [ ]:
nominal json_value =
    `(
        rust_type
            `($'serde_json_Value')
            `($'serde_json::Value')
            "type serde_json_Value = class end"
    )

### json_error

In [ ]:
nominal json_error =
    `(
        rust_type
            `($'serde_json_Error')
            `($'serde_json::Error')
            "type serde_json_Error = class end"
    )

### serde_wasm_bindgen_error

In [ ]:
nominal serde_wasm_bindgen_error =
    `(
        rust_type
            `($'serde_wasm_bindgen_Error')
            `($'serde_wasm_bindgen::Error')
            "type serde_wasm_bindgen_Error = class end"
    )

### js_string

In [ ]:
nominal js_string =
    `(
        rust_type
            `($'js_sys_JsString')
            `($'js_sys::JsString')
            "type js_sys_JsString = class end"
    )

### os_str

In [ ]:
nominal os_str =
    `(
        rust_type
            `($'std_ffi_OsStr')
            `($'std::ffi::OsStr')
            "type std_ffi_OsStr = class end"
    )

### c_str

In [ ]:
nominal c_str =
    `(
        rust_type
            `($'std_ffi_CStr')
            `($'std::ffi::CStr')
            "type std_ffi_CStr = class end"
    )

### c_string

In [ ]:
nominal c_string =
    `(
        rust_type
            `($'std_ffi_CString')
            `($'std::ffi::CString')
            "type std_ffi_CString = class end"
    )

### os_string

In [ ]:
nominal os_string =
    `(
        rust_type
            `($'std_ffi_OsString')
            `($'std::ffi::OsString')
            "type std_ffi_OsString = class end"
    )

### raw_string_literal

In [ ]:
inl raw_string_literal (s : string) : rust.ref str =
    backend_switch {
        Fsharp = fun () => !\($'"r\#\\"" + !s + "\\"\#"') : rust.ref str
        Rust = fun () => $'(&*!s)' : rust.ref str
    }

### raw_string_literal_static

In [ ]:
inl raw_string_literal_static (s : string) : rust.static_ref str =
    backend_switch {
        Fsharp = fun () => !\($'"r\#\\"" + !s + "\\"\#"') : rust.static_ref str
        Rust = fun () => $'Box::leak(String::from(&*!s).into_boxed_str())' : rust.static_ref str
    }

### (~#)

In [ ]:
inl (~#) s =
    s |> raw_string_literal

### (~###)

In [ ]:
inl (~###) s =
    s |> raw_string_literal_static

### include_str

In [ ]:
inl include_str (path : string) : rust.ref str =
    !\($'"include_str\!(\\\"" + !path + "\\\")"')

### as_str

In [ ]:
inl as_str (s : string) : rust.ref str =
    run_target_args (fun () => s) function
        | Rust => fun s => !\\(s, $'"&*$0"')
        | _ => convert.unbox

### from_iter

In [ ]:
inl from_iter forall (t : * -> *). (s : t char) : std_string =
    !\\(s, $'"String::from_iter($0)"')

### ref_to_std_string

In [ ]:
inl ref_to_std_string (str : rust.ref str) : std_string =
    run_target_args (fun () => str) function
        | Rust => fun str => !\\(str, $'"String::from($0)"')
        | _ => convert.unbox

### cow_to_std_string

In [ ]:
inl cow_to_std_string (str : rust.cow str) : std_string =
    !\\(str, $'"String::from($0)"')

### to_std_string

In [ ]:
inl to_std_string (s : string) : std_string =
    s |> as_str |> ref_to_std_string

### as_str_std

In [ ]:
inl as_str_std (s : std_string) : rust.ref str =
    !\\(s, $'"$0.as_str()"')

### into_boxed_str

In [ ]:
inl into_boxed_str (s : std_string) : rust.box str =
    !\\(s, $'"$0.into_boxed_str()"')

### os_string_as_ref

In [ ]:
inl os_string_as_ref (s : os_string) : rust.ref os_str =
    !\\(s, $'"$0.as_ref()"')

### to_os_string

In [ ]:
inl to_os_string (s : rust.ref os_str) : os_string =
    !\\(s, $'"$0.to_os_string()"')

### new_c_string

In [ ]:
inl new_c_string (s : std_string) : c_string =
    !\\(s, $'"std::ffi::CString::new($0).unwrap()"')

### os_to_str

In [ ]:
inl os_to_str (s : os_string) : optionm'.option' (rust.ref str) =
    !\\(s, $'"$0.to_str()"')

### from_os_str_ref

In [ ]:
inl from_os_str_ref s =
    s
    |> to_os_string
    |> os_to_str
    |> optionm'.unwrap
    |> ref_to_std_string
    |> from_std_string

### format_custom'

In [ ]:
inl format_custom' (f : string) x : std_string =
    run_target function
        | Rust => fun () =>
            !\\(x, $'"format\!(\\\"" + !f + "\\\", $0)"')
        | _ => fun () => null ()

### format_debug'

In [ ]:
inl format_debug' x : std_string =
    run_target function
        | Rust => fun () =>
            !\\(x, $'"format\!(\\\"{:?}\\\", $0)"')
        | _ => fun () => null ()

### format'

In [ ]:
inl format' x : std_string =
    run_target_args (fun () => x) function
        | Rust => fun x =>
            !\\(x, $'"format\!(\\\"{}\\\", $0)"')
        | _ => fun _ => null ()

### format_hex'

In [ ]:
inl format_hex' x : std_string =
    !\\(x, $'"format\!(\\\"{:02x}\\\", $0)"')

### format''

In [ ]:
inl format'' (format : string) : std_string =
    !\($'@@$"format\!(" + !format + ")"')

### regex

In [ ]:
nominal regex =
    `(
        rust_type
            `($'regex_Regex')
            `($'regex::Regex')
            "type regex_Regex = class end"
    )

### regex_error

In [ ]:
nominal regex_error =
    `(
        rust_type
            `($'regex_Error')
            `($'regex::Error')
            "type regex_Error = class end"
    )

### new_regex

In [ ]:
inl new_regex (pattern : string) : resultm.result' regex regex_error =
    !\\(pattern, $'$"regex::Regex::new(&$0)"')

### captures

In [ ]:
nominal regex_captures t =
    `(
        rust_type
            `($'regex_Captures<`t>')
            `($'regex::Captures<`t>')
            "type regex_Captures<'T> = class end"
    )

### regex_capture_matches

In [ ]:
nominal regex_capture_matches =
    `(
        rust_type
            `($'regex_CaptureMatches')
            `($'regex::CaptureMatches')
            "type regex_CaptureMatches = class end"
    )

### regex_capture_names

In [ ]:
nominal regex_capture_names =
    `(
        rust_type
            `($'regex_CaptureNames')
            `($'regex::CaptureNames')
            "type regex_CaptureNames = class end"
    )

inl regex_capture_names (regex : regex) : regex_capture_names =
    !\\(regex, $'$"$0.capture_names()"')

### match'

In [ ]:
nominal match' =
    `(
        rust_type
            `($'regex_Match')
            `($'regex::Match')
            "type regex_Match = class end"
    )

### regex_captures_iter

In [ ]:
inl regex_captures_iter (s : rust.static_ref (rust.mut' std_string)) (regex : regex) : regex_capture_matches =
    inl regex = regex |> rust.emit
    !\\(regex, $'$"$0.captures_iter(!s)"')

### regex_captures

In [ ]:
let regex_captures (s : string) (regex : regex) : am'.vec (mapm.hash_map string string) =

    inl s = s |> to_std_string
    fun () =>
        inl matches =
            inl s = s |> rust.new_box |> rust.box_leak
            regex |> regex_captures_iter s

        (!\($'"true; let _regex_captures : Vec<_> = !matches.map(|x| { //"') : bool) |> ignore

        inl fn (match' : rust.static_ref (rust.mut' (regex_captures rust.static_lifetime))) : mapm.hash_map string string =

            inl names = regex |> regex_capture_names

            (!\($'"true; let _regex_captures : std::collections::HashMap<_, _> = !names.map(|x| { //"') : bool) |> ignore

            inl fn (n : string) : pair string string =
                inl n' = n |> rust.clone

                new_pair n' !\\(n, $'$"!match'.name(&$0).map(|x| x.as_str()).unwrap_or(\\\"\\\").to_string().into()"')

            (!\\(fn !\($'"x.unwrap_or(\\\"\\\").to_string().into()"'), $'"true; $0 }).collect()"') : bool) |> ignore

            !\($'"_regex_captures"')

        inl x =
            !\($'$"x"')
            |> rust.new_box
            |> rust.box_leak

        (!\\(fn x, $'"true; $0 }).collect::<Vec<_>>()"') : bool) |> ignore

        !\($'"_regex_captures"')

    |> rust.capture_move

In [32]:
///- --test
///> rust -d regex

"fable-library-ts\\.(?<a>[\\d.]+)$"
|> new_regex
|> resultm.unwrap'
|> regex_captures "fable_modules/fable-library-ts.4.17.0"
|> am'.vec_map (mapm.to_vec >> am'.vec_sort_by_key id)
|> format_debug
|> _assert_eq (
    ;[
        ;[ "", ""; "a", "4.17.0" ]
        |> am'.to_vec
    ]
    |> am'.to_vec
    |> format_debug
)

__assert_eq { expected = [[("", ""), ("a", "4.17.0")]] }

### serialize

In [ ]:
inl serialize forall t. (x : t) : resultm.result' std_string json_error =
    !\($'"serde_json::to_string(&!x)"')

### deserialize

In [ ]:
inl deserialize forall t. (json : string) : resultm.result' t std_string =
    inl json = join json
    inl json = json |> as_str
    !\\(json, $'"serde_json::from_str(&$0)"')
    |> resultm.map_error' fun (x : json_error) => x |> format'

### borsh_serialize

In [ ]:
inl borsh_serialize forall t. (data : t) : am'.vec u8 =
    (!\($'"true; let mut data = Vec::new()"') : bool) |> ignore
    (!\\(data, $'"true; borsh::BorshSerialize::serialize(&$0, &mut data).unwrap()"') : bool) |> ignore
    !\($'"data"')

### borsh_deserialize

In [ ]:
inl borsh_deserialize forall t. (data : array_base u8) : resultm.result' t std_string =
    inl data = data |> am'.as_slice
    (!\($'"true; let mut !data = !data"') : bool) |> ignore
    inl result = !\($'"borsh::BorshDeserialize::deserialize(&mut !data)"')
    result
    |> resultm.map_error' fun (x : borsh_io_error) => x |> format'

### deserialize_vec

In [ ]:
inl deserialize_vec (value : json_value) : resultm.result' (am'.vec u8) std_string =
    inl value = join value
    !\($'"serde_json::from_value(!value)"')
    |> resultm.map_error' fun (x : json_error) => x |> format'

### encode_uri_component

In [ ]:
inl encode_uri_component (s : std_string) : js_string =
    !\($'"js_sys::encode_uri_component(&!s)"')

### strip_prefix

In [ ]:
inl strip_prefix (prefix : char) (s : std_string) : optionm'.option' (rust.ref str) =
    inl s = join s
    !\($'"!s.strip_prefix((!prefix) as char)"')

### str_from_utf8

In [ ]:
inl str_from_utf8 (bytes : rust.ref (am'.slice u8)) : resultm.result' (rust.ref str) utf8_error =
    !\\(bytes, $'"std::str::from_utf8($0)"')

### string_from_utf8

In [ ]:
inl string_from_utf8 (bytes : am'.vec u8) : resultm.result' std_string from_utf8_error =
    inl bytes = join bytes
    !\\(bytes, $'"std::string::String::from_utf8($0)"')

### base64_decode

In [ ]:
inl base64_decode (s : std_string) : result std_string std_string =
    fun () =>
        inl s = join s
        inl bytes : resultm.result' (am'.vec u8) base64_decode_error =
            !\($'"base64::Engine::decode(&base64::engine::general_purpose::STANDARD, !s)"')
        bytes
        |> resultm.map_error' format'
        |> resultm.try'
        |> string_from_utf8
        |> resultm.map_error' format'
    |> fun x =>
        join x ()
        |> resultm.unbox

### encoding'

In [ ]:
nominal encoding' =
    `(
        rust_type
            `($'encoding_rs_Encoding')
            `($'encoding_rs::Encoding')
            "type encoding_rs_Encoding = class end"
    )

### encoding_utf8'

In [ ]:
inl encoding_utf8' () : rust.ref encoding' =
    !\($'"encoding_rs::UTF_8"')

### encoding_1252

In [ ]:
inl encoding_1252' () : rust.ref encoding' =
    !\($'"encoding_rs::WINDOWS_1252"')

### encoding_encode

In [ ]:
inl encoding_encode' (encoding : rust.ref encoding') (text : string) : rust.cow (am'.slice u8) =
    !\\((encoding, text), $'"$0.encode(&*$1).0"')

### utf8_decode

In [ ]:
inl utf8_decode (data : am'.vec u8) : resultm.result' std_string (rust.cow str) =
    !\($'$"encoding::Encoding::decode(encoding::all::UTF_8, &!data, encoding::DecoderTrap::Replace)"')

### windows

In [ ]:
nominal windows t =
    `(
        rust_type
            `($'std_slice_Windows<`t>')
            `($'std::slice::Windows<`t>')
            "type std_slice_Windows<'T> = class end"
    )

inl windows (len : convert.unativeint) (source : am'.vec u8) : windows u8 =
    inl source = source |> rust.new_box |> rust.box_leak
    inl result = !\\(len, $'"<[_]>::windows(!source, $0)"')
    result

### any

In [ ]:
inl any forall t. (fn : string -> bool) (source : windows t) : bool =
    (!\($'"true; let mut !source = !source"') : bool) |> ignore
    inl fn' x =
        x
        |> str_from_utf8
        |> resultm.unwrap_or' #""
        |> ref_to_std_string
        |> from_std_string
        |> fn
    !\\(fn', $'"!source.any(move |x| $0(x))"')

### slice_contains

In [ ]:
inl slice_contains (text : string) (source : am'.vec u8) : bool =
    fun () =>
        inl source = join source
        source
        |> windows (text |> length |> (fun x => x : i32) |> convert.convert)
        |> any ((=.) text)
    |> fun x => join x ()

### as_bytes

In [ ]:
inl as_bytes (text : string) : rust.ref (am'.slice u8) =
    inl text = join text
    !\($'"!text.as_bytes()"')

### into_bytes

In [ ]:
inl into_bytes (x : std_string) : am'.vec u8 =
    backend_switch {
        Rust = fun () => $'(!x).as_bytes().to_vec()' : am'.vec u8
    }

## python

### encode_utf8

In [ ]:
inl encode_utf8 (s : string) : string =
    inl encoding = "utf-8"
    backend_switch {
        Python = fun () =>
            $'!s.encode(!encoding)' : string
    }

## sm'

### contains

In [ ]:
inl contains (value : string) (s : string) : bool =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/string"
            $'!s |> string.contains(!value)' : bool
        Lua = fun () => $'string.find(!s, !value, 1, true) ~= nil ' : bool
        Fsharp = fun () => $'!s.Contains !value ' : bool
        Python = fun () => $'!value in !s ' : bool
        Rust = fun () => $'!s.contains(&*!value)' : bool
        TypeScript = fun () => $'!s.includes(!value)' : bool
        Zig = fun () => $'(std.mem.indexOf(u8, !s, !value) \!= null)' : bool
        Lean = fun () => $'(!s).contains !value ' : bool
    }

### to_string result t u

In [ ]:
instance to_string result t u = fun x =>
    real
        open rust
        typecase (t * u) with
        | string * string =>
            match x with
            | Ok x => x
            | Error x => "sm'.to_string result / Error: " ++# x
        | std_string * std_string =>
            match x with
            | Ok x => from_std_string x
            | Error x => "sm'.to_string result / Error: " ++# from_std_string x
        | _ => obj_to_string `u x

### format_exception

In [ ]:
inl format_exception (ex : exn) : string =
    run_target function
        | Fsharp => fun () => $'$"{!ex.GetType ()}: {!ex.Message}"'
        | Rust => fun () => $'!ex ' : string
        | TypeScript => fun () => $'String(!ex)' : string
        | _ => fun () => ex |> format_debug

In [33]:
///- --test --no-agree "exception text is backend-specific"
///> fsharp
///> cuda
///> rust
///> typescript

fun () => failwith "test"
|> _throws
|> optionm.value
|> format_exception
|> _assert_eq (run_target function
    | Fsharp _ => fun () => join "System.Exception: test"
    | Cuda _ => fun () => "test"
    | Rust => fun () => "test"
    | TypeScript _ => fun () => "Error: test"
    | _ => fun () => null ()
)

.fsx (fsharp)
__assert_eq { expected = System.Exception: test }
.py (cuda)
__assert_eq { expected = test }
.rs (rust)
__assert_eq { expected = test }
.ts (typescript)
__assert_eq { expected = Error: test }

0

### range

In [ ]:
inl range forall t. (start : am'.range t) (end : am'.range t) s =
    inl is_start, start, end =
        inl len () : int =
            real
                typecase t with
                | int => length `t s
                | _ => length' `t s
        match start, end with
        | Start start, End fn => true, start, len >> convert.to |> fn
        | End start_fn, End end_fn => false, start_fn (len >> convert.to), end_fn (len >> convert.to)
    inl start' = start |> convert.try_convert |> optionm.value : int
    inl end' = end |> convert.try_convert |> optionm.value : int
    inl start_is_var = start |> var_is
    inl same_fixed = start = end && (start_is_var |> not) && (end |> var_is |> not)
    inl start_var_s_fixed = start_is_var && (s |> var_is |> not)
    inl trace () =
        [
            "sm'.range"
            " / (s |> var_is): "; s |> var_is |> obj_to_string
            " // start: "; start |> obj_to_string
            " / (start |> var_is): "; start_is_var |> obj_to_string
            " // end: "; end |> obj_to_string
            " / (end |> var_is): "; end |> var_is |> obj_to_string
            " // same_fixed: "; same_fixed |> obj_to_string
            " / start_var_s_fixed: "; start_var_s_fixed |> obj_to_string
        ]
        |> concat_list ""
    backend_switch {
        Gleam = fun () =>
            ()
        Lua = id : () -> ()
        Fsharp = id : () -> ()
        Python = id : () -> ()
        Rust = id : () -> ()
        TypeScript = id : () -> ()
        Zig = id : () -> ()
    }
    s |> slice start' (end' - (if is_start && (same_fixed || start_var_s_fixed) then 0 else 1))

In [34]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"abcde"
|> range (am'.Start 0i32) (am'.End eval)
|> _assert_eq "abcde"

(join "fghij")
|> range (am'.Start 1i32) (am'.End eval)
|> _assert_eq "ghij"

(join "klmno")
|> range (am'.Start 1i32) (am'.End fun _ => 1)
|> _assert_eq "l"

"pqrst"
|> range (am'.Start 1i32) (am'.End fun _ => 1)
|> _assert_eq "q"

"uvwxy"
|> range (am'.Start (join 1i32)) (am'.End fun _ => 1)
|> _assert_eq "v"

"zabcd"
|> range (am'.Start (join 1i32)) (am'.End fun _ => join 1)
|> _assert_eq "a"

(join "efghi")
|> range (am'.Start 1i32) (am'.End fun _ => 5)
|> _assert_eq "fghi"

"jklmn"
|> range (am'.End fun x => 1i32) (am'.End fun _ => join 5)
|> _assert_eq "klmn"

"opqrs"
|> range (am'.End fun x => x () - 4i32) (am'.End fun x => x () - 1i32)
|> _assert_eq "pqr"

(join "stuvw")
|> range (am'.End fun x => x () - 4i32) (am'.End fun x => x () - 1)
|> _assert_eq "tuv"

(join "wxyza")
|> range (am'.Start (join 2i32)) (am'.End fun x => x () - 2)
|> _assert_eq "y"

(join "bcdef")
|> range (am'.Start (join 2i32)) (am'.End fun x => x () - 3)
|> _assert_eq ""

(join "ghijk")
|> range (am'.Start (join 1i32)) (am'.End fun x => x () - 3)
|> _assert_eq "h"

(join "lmnop")
|> range (am'.Start (join 1i32)) (am'.End fun x => x () - 4)
|> _assert_eq ""

(join "qrstu")
|> range (am'.End fun x => x () - 4i32) (am'.End fun _ => 4)
|> _assert_eq "rst"

"vwxyz"
|> range (am'.End fun x => join x () - 4i32) (am'.End fun _ => 4)
|> _assert_eq "wxy"

.gleam (gleam)
__assert_eq { expected = abcde }
__assert_eq { expected = ghij }
__assert_eq { expected = l }
__assert_eq { expected = q }
__assert_eq { expected = v }
__assert_eq { expected = a }
__assert_eq { expected = fghi }
__assert_eq { expected = klmn }
__assert_eq { expected = pqr }
__assert_eq { expected = tuv }
__assert_eq { expected = y }
__assert_eq { expected =  }
__assert_eq { expected = h }
__assert_eq { expected =  }
__assert_eq { expected = rst }
__assert_eq { expected = wxy }
.fsx (fsharp)
__assert_eq { expected = abcde }
__assert_eq { expected = ghij }
__assert_eq { expected = l }
__assert_eq { expected = q }
__assert_eq { expected = v }
__assert_eq { expected = a }
__assert_eq { expected = fghi }
__assert_eq { expected = klmn }
__assert_eq { expected = pqr }
__assert_eq { expected = tuv }
__assert_eq { expected = y }
__assert_eq { expected =  }
__assert_eq { expected = h }
__assert_eq { expected =  }
__assert_eq { expected = rst }
__assert_eq { expected = wxy }
.py (

0

### ellipsis_end

In [ ]:
let ellipsis_end (max : i64) (s : string) =
    inl len = s |> length
    if len <= max
    then s
    else
        inl half = (max |> convert.f64) / 2
        inl start_half = half |> math.ceil |> convert.i64
        inl end_half = half |> math.floor |> convert.i64
        inl start = s |> slice 0 (start_half - 1)
        inl end = s |> slice (len - end_half) (len - 1)
        [ start; "..."; end ]
        |> concat_list ""

In [35]:
///- --test
///> gleam
///> lua
///> fsharp

"12345"
|> ellipsis_end 2
|> _assert_eq "1...5"

"12345"
|> ellipsis_end 3
|> _assert_eq "12...5"

"1234567"
|> ellipsis_end 4
|> _assert_eq "12...67"

.gleam (gleam)
__assert_eq { expected = 1...5 }
__assert_eq { expected = 12...5 }
__assert_eq { expected = 12...67 }
.lua (lua)
__assert_eq { expected = 1...5 }
__assert_eq { expected = 12...5 }
__assert_eq { expected = 12...67 }
.fsx (fsharp)
__assert_eq { expected = 1...5 }
__assert_eq { expected = 12...5 }
__assert_eq { expected = 12...67 }

### format_ellipsis

In [ ]:
inl format_ellipsis s =
    s
    |> format_debug
    |> ellipsis_end 400

### replace_regex

In [ ]:
let replace_regex (pattern : string) (replacement : string) (s : string) : string =
    run_target_args (fun () => s, pattern, replacement) function
        | Fsharp => fun s, pattern, replacement =>
            $'System.Text.RegularExpressions.Regex.Replace (!s, !pattern, !replacement)'
        | Rust => fun s, pattern, replacement =>
            inl regex = pattern |> new_regex |> resultm.unwrap'
            inl s = join s
            !\\((regex, s, replacement), $'$"$0.replace_all(&*$1, &*$2)"')
            |> cow_to_std_string
            |> from_std_string
        | Gleam => fun s, pattern, replacement =>
            global "import gleam/regexp"
            $'let assert Ok(re) = !pattern |> regexp.from_string'
            $'re |> regexp.replace(!s, !replacement)' : string
        | _ => fun _ => join ""

In [36]:
///- --test
///> fsharp
///> gleam
///> rust -d regex

" 123"
|> replace_regex "\\s\\w2" ""
|> _assert_eq "3"

.fsx (fsharp)
__assert_eq { expected = 3 }
.gleam (gleam)
__assert_eq { expected = 3 }
.rs (rust)
__assert_eq { expected = 3 }

In [37]:
///- --test
///> rust -d regex

"    let main args =\n        ()\n"
|> replace_regex "(?P<a> *)(?P<b>let\\s+main\\s+.*?\\s*=)" "$a[<EntryPoint>]\n$a$b"
|> _assert_eq "    [<EntryPoint>]\n    let main args =\n        ()\n"

__assert_eq { expected =     [<EntryPoint>]
    let main args =
        ()
 }

## main

In [ ]:
///> fsharp

inl main () =
    $'let contains x = !contains x' : ()
    $'let ends_with x = !ends_with x' : ()
    $'let pad_left x = !pad_left x' : ()
    $'let pad_right x = !pad_right x' : ()
    $'let replace x = !replace x' : ()
    $'let replace_regex x = !replace_regex x' : ()
    inl slice (a : i32) (b : i32) c = slice a b c
    $'let slice x = !slice x' : ()
    $'let split x = !split x' : ()
    $'let split_string x = !split_string x' : ()
    $'let starts_with x = !starts_with x' : ()
    $'let substring x = !substring x' : ()
    $'let to_lower x = !to_lower x' : ()
    $'let to_upper x = !to_upper x' : ()
    $'let trim x = !trim x' : ()
    $'let ellipsis x = !ellipsis x' : ()
    $'let ellipsis_end x = !ellipsis_end x' : ()
    $'let format_exception x = !format_exception x' : ()
    $'let concat_array x = !concat_array x' : ()
    inl concat a (b : seq.seq' string) = concat a b
    $'let concat x = !concat x' : ()
    $'let join\' x = !join' x' : ()
    $'let to_char_array x = !to_char_array x' : ()

## rust

### to_string std_string

In [38]:
open rust
instance to_string std_string = from_std_string